# Data Collaboration クイックスタート

`main.py` を経由せずに `experiments/experiment.py` と `src/data_collaboration.py` を直接呼び出し、データコラボレーション (DC) の流れを最小構成で確認するための教科書的な手順をまとめています。

## Step 1: パス設定とモジュールの準備

リポジトリ直下でノートブックを実行していることを確認し、Python のモジュール探索パスにプロジェクトルートを追加します。

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
print(f'Project root: {PROJECT_ROOT}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


Project root: C:\Users\sueya\Git-Repositories\takano_labo\dca


## Step 2: Config オブジェクトの組み立て

`Config` はキーワード引数を辞書のように保持する軽量クラスです。ここではアンカーデータ生成方法 (`anchor_method`)、推論器 (`h_model`)、保存先 (`output_path`) など DC 実行に必要な最小限のパラメータをまとめて設定します。

In [9]:
from config.config import Config
from pprint import pprint
from src.paths import OUTPUT_DIR

output_dir = OUTPUT_DIR / 'quickstart_notebook'
output_dir.mkdir(parents=True, exist_ok=True)

base_config = {
    'name': 'tutorial_digits_imakura',
    'dataset': 'digits',
    'y_name': 'target',
    'num_institution': 3,
    'num_institution_user': 200,
    'data_distribution': 'even',
    'num_anchor_data': 256,
    'dim_intermediate': 20,
    'dim_integrate': 10,
    'F_type': 'svd',
    'G_type': 'Imakura',
    'metrics': 'accuracy',
    'seed': 42,
    'visualize': False,
    'load_df_data': False,
    'load_intermediate_data': False,
    'inter_normalization': False,
    'anchor_method': 'gaussian',
    'output_path': output_dir,
    'h_model': 'mlp',
}

tutorial_config = Config(**base_config)
pprint(tutorial_config.__dict__)


{'F_type': 'svd',
 'G_type': 'Imakura',
 'anchor_method': 'gaussian',
 'data_distribution': 'even',
 'dataset': 'digits',
 'dim_integrate': 10,
 'dim_intermediate': 20,
 'h_model': 'mlp',
 'inter_normalization': False,
 'load_df_data': False,
 'load_intermediate_data': False,
 'metrics': 'accuracy',
 'name': 'tutorial_digits_imakura',
 'num_anchor_data': 256,
 'num_institution': 3,
 'num_institution_user': 200,
 'output_path': WindowsPath('c:/Users/sueya/Git-Repositories/takano_labo/dca/output/quickstart_notebook'),
 'seed': 42,
 'visualize': False,
 'y_name': 'target'}


## Step 3: `experiments.experiment.run_once` を直接呼び出す

`run_once` は `main.py` から呼ばれているワンショット実行関数です。設定とロガーを渡すだけで、データ読み込みから DC の評価までを一度に試せます。

In [10]:
import logging
from experiments.experiment import run_once

logger = logging.getLogger('dc_tutorial')
if not logger.handlers:
    handler = logging.StreamHandler()
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

result = run_once(tutorial_config, logger)
print('run_once result:', result)


データセット:digits
データ新規読み込み中...
Index(['pixel_0_0', 'pixel_0_1', 'pixel_0_2', 'pixel_0_3', 'pixel_0_4',
       'pixel_0_5', 'pixel_0_6', 'pixel_0_7', 'pixel_1_0', 'pixel_1_1',
       'pixel_1_2', 'pixel_1_3', 'pixel_1_4', 'pixel_1_5', 'pixel_1_6',
       'pixel_1_7', 'pixel_2_0', 'pixel_2_1', 'pixel_2_2', 'pixel_2_3',
       'pixel_2_4', 'pixel_2_5', 'pixel_2_6', 'pixel_2_7', 'pixel_3_0',
       'pixel_3_1', 'pixel_3_2', 'pixel_3_3', 'pixel_3_4', 'pixel_3_5',
       'pixel_3_6', 'pixel_3_7', 'pixel_4_0', 'pixel_4_1', 'pixel_4_2',
       'pixel_4_3', 'pixel_4_4', 'pixel_4_5', 'pixel_4_6', 'pixel_4_7',
       'pixel_5_0', 'pixel_5_1', 'pixel_5_2', 'pixel_5_3', 'pixel_5_4',
       'pixel_5_5', 'pixel_5_6', 'pixel_5_7', 'pixel_6_0', 'pixel_6_1',
       'pixel_6_2', 'pixel_6_3', 'pixel_6_4', 'pixel_6_5', 'pixel_6_6',
       'pixel_6_7', 'pixel_7_0', 'pixel_7_1', 'pixel_7_2', 'pixel_7_3',
       'pixel_7_4', 'pixel_7_5', 'pixel_7_6', 'pixel_7_7', 'target'],
      dtype='object')
****************

100%|██████████| 3/3 [00:00<00:00, 52.53it/s]
中間表現（訓練データ）の数と次元数: (200, 20)
統合表現（訓練データ）の数と次元数: (600, 10)


中間表現の次元数:  20
num_row 256 num_col 64
Xs_train[0].shape (200, 64) Xs_test[0].shape (200, 64)
********************統合表現の生成********************
擬似逆行列の絶対値の総和: 6.09966083097232
統合表現の次元数:  10


{'inter': 0.0, 'integ': 0.0, 'inter_test': 0.0, 'integ_test': 0.0}


[LNI] inter per-institution: [0.0000, 0.0000, 0.0000]
[LNI] integ per-institution: [0.0000, 0.0000, 0.0000]
[LNI] inter_test per-institution: [0.0000, 0.0000, 0.0000]
[LNI] integ_test per-institution: [0.0000, 0.0000, 0.0000]


提案手法の評価値: 0.8300
提案手法の評価値: 0.8850
提案手法の評価値: 0.8100
機関ごとの accuracy: [0.83, 0.885, 0.81]
平均: 0.841667, 最小: 0.810000, 最大: 0.885000


評価値2 0.8416666666666667
機関ごとの accuracy: [0.83, 0.885, 0.81]
平均: 0.8417, 最小: 0.8100, 最大: 0.8850
run_once result: 0.8416666666666667


## Step 4: `DataCollaborationAnalysis` を直接操作する

内部処理を細かく確認したい場合は、`DataCollaborationAnalysis` を自分で初期化して `run()` を呼び出します。データの準備とクラス初期化の流れを段階的に追いかけられます。

In [11]:
from src.load_data import load_data
from src.institution_data import prepare_institutional_dataset
from src.data_collaboration import DataCollaborationAnalysis

manual_config = Config(**base_config)

df = load_data(config=manual_config)
Xs_train, Xs_test, ys_train, ys_test, train_df, test_df = prepare_institutional_dataset(df, manual_config)

dc = DataCollaborationAnalysis(
    config=manual_config,
    logger=logger,
    train_df=train_df,
    test_df=test_df,
    Xs_train=Xs_train,
    Xs_test=Xs_test,
    ys_train=ys_train,
    ys_test=ys_test,
)
dc.run()

print('integrated train shape:', dc.X_train_integ.shape)
print('integrated test shape:', dc.X_test_integ.shape)
print('integrated label shape:', dc.y_train_integ.shape)
print('integrated test label shape:', dc.y_test_integ.shape)


Index(['pixel_0_0', 'pixel_0_1', 'pixel_0_2', 'pixel_0_3', 'pixel_0_4',
       'pixel_0_5', 'pixel_0_6', 'pixel_0_7', 'pixel_1_0', 'pixel_1_1',
       'pixel_1_2', 'pixel_1_3', 'pixel_1_4', 'pixel_1_5', 'pixel_1_6',
       'pixel_1_7', 'pixel_2_0', 'pixel_2_1', 'pixel_2_2', 'pixel_2_3',
       'pixel_2_4', 'pixel_2_5', 'pixel_2_6', 'pixel_2_7', 'pixel_3_0',
       'pixel_3_1', 'pixel_3_2', 'pixel_3_3', 'pixel_3_4', 'pixel_3_5',
       'pixel_3_6', 'pixel_3_7', 'pixel_4_0', 'pixel_4_1', 'pixel_4_2',
       'pixel_4_3', 'pixel_4_4', 'pixel_4_5', 'pixel_4_6', 'pixel_4_7',
       'pixel_5_0', 'pixel_5_1', 'pixel_5_2', 'pixel_5_3', 'pixel_5_4',
       'pixel_5_5', 'pixel_5_6', 'pixel_5_7', 'pixel_6_0', 'pixel_6_1',
       'pixel_6_2', 'pixel_6_3', 'pixel_6_4', 'pixel_6_5', 'pixel_6_6',
       'pixel_6_7', 'pixel_7_0', 'pixel_7_1', 'pixel_7_2', 'pixel_7_3',
       'pixel_7_4', 'pixel_7_5', 'pixel_7_6', 'pixel_7_7', 'target'],
      dtype='object')
********************中間表現の生成*****************

100%|██████████| 3/3 [00:00<00:00, 49.49it/s]
中間表現（訓練データ）の数と次元数: (200, 20)
統合表現（訓練データ）の数と次元数: (600, 10)


中間表現の次元数:  20
num_row 256 num_col 64
Xs_train[0].shape (200, 64) Xs_test[0].shape (200, 64)
********************統合表現の生成********************
擬似逆行列の絶対値の総和: 6.09966083097232
統合表現の次元数:  10
[LNI] inter per-institution: [0.0000, 0.0000, 0.0000]
[LNI] integ per-institution: [0.0000, 0.0000, 0.0000]
[LNI] inter_test per-institution: [0.0000, 0.0000, 0.0000]
[LNI] integ_test per-institution: [0.0000, 0.0000, 0.0000]


{'inter': 0.0, 'integ': 0.0, 'inter_test': 0.0, 'integ_test': 0.0}


integrated train shape: (600, 10)
integrated test shape: (600, 10)
integrated label shape: (600,)
integrated test label shape: (600,)


## Step 5: 評価指標を個別に計算する

統合表現が得られたら、`dca_analysis` を使って各機関ごとの指標を自分で算出することもできます。

In [12]:
import numpy as np
from src.institutional_analysis import dca_analysis

scores = []
for idx in range(manual_config.num_institution):
    score = dca_analysis(
        X_train_integ=dc.X_train_integ,
        X_test_integ=dc.Xs_test_integ[idx],
        y_train_integ=dc.y_train_integ,
        y_test_integ=dc.ys_test_integ[idx],
        config=manual_config,
        logger=logger,
    )
    scores.append(score)

scores_array = np.array(scores, dtype=float)
print('per-institution scores:', scores_array)
print('mean score:', np.nanmean(scores_array))


提案手法の評価値: 0.8300
提案手法の評価値: 0.8850
提案手法の評価値: 0.8100


per-institution scores: [0.83  0.885 0.81 ]
mean score: 0.8416666666666667


## Step 6: 応用のヒント

- `base_config` の `F_type`, `G_type`, 次元数を変更して比較を試す
- `load_df_data` や `load_intermediate_data` を `True` にしてデータ分割・中間表現を保存／再利用する
- `metrics` を `accuracy` 以外（例: `auc`）に変えて評価指標を切り替える
- `data_distribution` を `division` や `semi` にして分割方法を検証する